# Clickbait Title/Body Transformer Training (Korean DeBERTa Base)
`title/body` split CSV를 사용해 `team-lucid/deberta-v3-base-korean` 계열의 DeBERTa를 파인튜닝합니다. 산출물은 별도 run으로 저장합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/sw_project/ai_features/clickbait_detection'
TRANSFORMER_SRC = os.path.join(DRIVE_ROOT, 'clickbait_transformer_finetune')
TRANSFORMER_WORK = '/content/clickbait_transformer_finetune'
DATA_SRC = os.path.join(DRIVE_ROOT, 'data')

assert os.path.exists(TRANSFORMER_SRC), f'Not found: {TRANSFORMER_SRC}'
assert os.path.exists(DATA_SRC), f'Not found: {DATA_SRC}'

if os.path.exists(TRANSFORMER_WORK):
    shutil.rmtree(TRANSFORMER_WORK)
shutil.copytree(TRANSFORMER_SRC, TRANSFORMER_WORK)
if os.path.exists(os.path.join(TRANSFORMER_WORK, 'data')):
    shutil.rmtree(os.path.join(TRANSFORMER_WORK, 'data'))
shutil.copytree(DATA_SRC, os.path.join(TRANSFORMER_WORK, 'data'), dirs_exist_ok=True)

os.chdir(TRANSFORMER_WORK)
!pip -q install -r requirements-colab.txt


In [ ]:
!curl -L -o train_transformer.py https://raw.githubusercontent.com/arimmiii/2026-1-CSC4004-1-team07/AI_features/ai_features/clickbait_detection/clickbait_transformer_finetune/train_transformer.py
!python3 -m py_compile train_transformer.py


In [ ]:
!python3 -u train_transformer.py \
  --model-name team-lucid/deberta-v3-base-korean \
  --train-data data/train.csv \
  --valid-data data/valid.csv \
  --test-data data/test.csv \
  --title-col title \
  --body-col body \
  --label-col label \
  --max-length 128 \
  --epochs 2 \
  --batch-size 4 \
  --learning-rate 1e-5 \
  --warmup-ratio 0.1 \
  --gradient-accumulation-steps 2 \
  --output-dir outputs/deberta_v3_base_title_body_run1 \
  --save-model-dir models/deberta_v3_base_title_body_run1


In [ ]:
!ls -lah models | sed -n '1,120p'


### Note
`team-lucid/deberta-v3-base-korean`는 `roberta-base`보다 무거울 수 있습니다. Colab에서 OOM이 나면 `--batch-size 2`로 낮추세요.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = "models/deberta_v3_base_title_body_run1"
metrics_path = Path(model_dir) / "metrics.json"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def load_split(csv_path):
    df = pd.read_csv(csv_path, usecols=["title", "body", "label"])
    texts = (
        df["title"].fillna("").astype(str).str.strip()
        + " [SEP] "
        + df["body"].fillna("").astype(str).str.strip()
    ).str.strip().tolist()
    labels = df["label"].astype(int).to_numpy()
    return texts, labels

def collate_fn(batch_texts):
    return tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )

results = {}
for split_name, csv_path in [
    ("valid", "data/valid.csv"),
    ("test", "data/test.csv"),
]:
    texts, labels = load_split(csv_path)
    dataloader = DataLoader(texts, batch_size=32, shuffle=False, collate_fn=collate_fn)

    preds = []
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())

    preds = np.asarray(preds)
    results[split_name] = {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
        "weighted_f1": float(f1_score(labels, preds, average="weighted")),
    }

metrics_path.parent.mkdir(parents=True, exist_ok=True)
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"saved -> {metrics_path}")
print(json.dumps(results, ensure_ascii=False, indent=2))
